In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...
remote: Enumerating objects: 785, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 785 (delta 42), reused 0 (delta 0), pack-reused 708 (from 1)
Receiving objects: 100% (785/785), 27.99 MiB | 31.39 MiB/s, done.
Resolving deltas: 100% (410/410), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.5 MB/s eta 0:00:00


In [3]:
! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorizationImpressions_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [5]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [6]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [7]:
db_name = 'ease_study.db'
json_name = 'ease_best_params.json'

config = {
    'tune_parameters': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name': json_name,
    'copy_prev_best_params': False,
    'save_github': True,
    'candidates': '50',
    'n_trials': 100
}

In [8]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [9]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [10]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [11]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [12]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

In [13]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [14]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

All rows from table 'studies':
(1, 'ease_Optimization')


In [15]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_ease_r(optuna_trial):
    recommender_instance = EASE_R_Recommender(URM_train)
    
    params = {
        # Using the ranges from the latest trials in better_tuning_EASE_R.db
        'topK': optuna_trial.suggest_int("topK", 210, 390), # Old: [10, 1000]
        'l2_norm': optuna_trial.suggest_float("l2_norm", 70.0, 130.0), # Old: [1.0, 1000.0]
        'normalize_matrix': False, # Old: [False, True]
    }
    
    recommender_instance.fit(**params)
    
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [16]:
%%time

if config['tune_parameters']:
    study = optuna.create_study(direction='maximize', study_name='ease_fine_tuning_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_ease_r, n_trials=config['n_trials'])

[I 2025-12-22 21:23:36,028] A new study created in RDB with name: ease_fine_tuning_Optimization


Previous trials found: 0
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 18.95 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 24.31 sec. Users per second: 1113


[I 2025-12-22 21:24:23,156] Trial 0 finished with value: 0.38201311849106706 and parameters: {'topK': 240, 'l2_norm': 70.56792340018605}. Best is trial 0 with value: 0.38201311849106706.


EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 18.50 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 23.89 sec. Users per second: 1133


[I 2025-12-22 21:25:09,362] Trial 1 finished with value: 0.39445690463377114 and parameters: {'topK': 243, 'l2_norm': 91.2351251022063}. Best is trial 1 with value: 0.39445690463377114.


...

EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 18.34 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 24.15 sec. Users per second: 1121


[I 2025-12-22 22:43:31,444] Trial 98 finished with value: 0.41369139110966086 and parameters: {'topK': 356, 'l2_norm': 126.30687347499226}. Best is trial 76 with value: 0.41507093081529317.


EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 18.46 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 24.07 sec. Users per second: 1124


[I 2025-12-22 22:44:19,390] Trial 99 finished with value: 0.41354856422526887 and parameters: {'topK': 370, 'l2_norm': 123.92069687546184}. Best is trial 76 with value: 0.41507093081529317.


CPU times: user 1h 41min 37s, sys: 2min 56s, total: 1h 44min 33s
Wall time: 1h 20min 45s


In [17]:
if config['save_github'] and config['tune_parameters']:
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )

    if config['save_json']:
        j_name = config['json_name']
        l_path = f"/kaggle/working/{j_name}"
        r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
        
        best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
        
        try:
            for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
                if s.best_trial and s.best_trial.value > best_val:
                    best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
        except:
            best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name
    
        prev_val, exists = -float('inf'), False
        try:
            prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
            exists = True
        except:
            pass
    
        if not exists or best_val > prev_val:
            print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
            with open(l_path, 'w') as f:
                json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
            if config.get('save_github'):
                upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
        else:
            print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

No update needed. Best: 0.440152 <= Saved: 0.440152
